# 03. Препроцессинг и фиксированный train/test split

**Цель:** подготовить полностью готовые наборы для обучения моделей, в формате (`X_train`, `X_test`, `y_train`, `y_test`) parquet, а так же сохранить препроцессор (joblib), который будут использовать все модели, в том числе и в продакшенею

**Шаги препроцессинга:**
1) Загрузка датасета `nfs_2023_nte_all.parquet`
2) Формирование `X` из 42 признаков, `y = target` (бинаризация)
3) **Stratified split 70/30** с `random_state=42`. Этот split фиксируем и переиспользуем во всех последующих ноутбуках.
4) **Препроцессинг - fit ТОЛЬКО на train**
    - `log1p` для skewed-колонок (skewness > 2 на train),
    - `RobustScaler` для 41 числового признака,
    - `protocol` — не трансформируется.

5) Сохранение всех артефактов в `cache/` и `artifacts/`.

**Результаты (сохранения):** 
1) `X_train.parquet`, `X_test.parquet`
2) `y_train.parquet`, `y_test.parquet`
3) `preprocessor.joblib` — sklearn Pipeline (log1p + RobustScaler)
4) `preprocessing_config.json` — список логарифмируемых колонок, параметры скейлера, версия sklearn
5) `feature_names.json` — финальные 42 имени в порядке столбцов

In [1]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, RobustScaler

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------- Пути -------
try:
    NB_DIR = Path(__file__).resolve().parent
except NameError:
    NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR

CACHE_DIR  = PROJECT_ROOT / "cache"
ARTIFACTS  = PROJECT_ROOT / "artifacts"
TAB_DIR    = PROJECT_ROOT / "results" / "tables"
for p in (CACHE_DIR, ARTIFACTS, TAB_DIR):
    p.mkdir(parents=True, exist_ok=True)

LABEL_COL = "label"
BENIGN_LABEL = "BENIGN"
TEST_SIZE = 0.30

print("sklearn:", sklearn.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)

sklearn: 1.8.0
PROJECT_ROOT: /workspace


In [2]:
MODEL_FEATURES: list[str] = [
    "protocol",

    "src2dst_packets",
    "dst2src_packets",
    "src2dst_bytes",
    "dst2src_bytes",

    "bidirectional_duration_ms",

    "bidirectional_min_ps",
    "bidirectional_max_ps",
    "bidirectional_mean_ps",
    "bidirectional_stddev_ps",

    "src2dst_max_ps",
    "src2dst_min_ps",
    "src2dst_mean_ps",
    "src2dst_stddev_ps",

    "dst2src_max_ps",
    "dst2src_min_ps",
    "dst2src_mean_ps",
    "dst2src_stddev_ps",

    "bidirectional_mean_piat_ms",
    "bidirectional_stddev_piat_ms",
    "bidirectional_max_piat_ms",
    "bidirectional_min_piat_ms",

    "src2dst_mean_piat_ms",
    "src2dst_stddev_piat_ms",
    "src2dst_max_piat_ms",
    "src2dst_min_piat_ms",

    "dst2src_mean_piat_ms",
    "dst2src_stddev_piat_ms",
    "dst2src_max_piat_ms",
    "dst2src_min_piat_ms",

    "bidirectional_fin_packets",
    "bidirectional_syn_packets",
    "bidirectional_rst_packets",
    "bidirectional_psh_packets",
    "bidirectional_ack_packets",
    "bidirectional_urg_packets",
    "bidirectional_cwr_packets",
    "bidirectional_ece_packets",

    "src2dst_psh_packets",
    "dst2src_psh_packets",
    "src2dst_urg_packets",
    "dst2src_urg_packets",
]
assert len(MODEL_FEATURES) == 42
assert len(set(MODEL_FEATURES)) == 42, "В MODEL_FEATURES есть дубликаты"
PROTOCOL_COL = "protocol"
NUMERIC_FEATURES = [c for c in MODEL_FEATURES if c != PROTOCOL_COL]
assert len(NUMERIC_FEATURES) == 41
print(f"MODEL_FEATURES: 42 ({len(NUMERIC_FEATURES)} numeric + 1 protocol)")

MODEL_FEATURES: 42 (41 numeric + 1 protocol)


In [3]:
CACHE_PARQUET = CACHE_DIR / "nfs_2023_nte_all.parquet"

df = pd.read_parquet(CACHE_PARQUET, columns=MODEL_FEATURES + [LABEL_COL])
print(f"Загружено: {df.shape}")

# Бинарный target
y = (df[LABEL_COL].astype(str).str.strip().str.upper() != BENIGN_LABEL).astype(np.int64)
X = df[MODEL_FEATURES].copy()

# Приводим всё к float64 для корректной работы скейлера и log1p.
# protocol — целое, но в DataFrame будет float64
X = X.astype(np.float64)

print(f"X: {X.shape}, y: {y.shape}")
print(f"y == 1 (anomaly): {int(y.sum()):,} ({100 * y.mean():.2f}%)")
print(f"y == 0 (benign):  {int((y == 0).sum()):,}")

Загружено: (2111131, 43)
X: (2111131, 42), y: (2111131,)
y == 1 (anomaly): 498,864 (23.63%)
y == 0 (benign):  1,612,267


## 1. Stratified split 70/30

- `random_split=42`
- **Это единственное место где делается train/test split.** Все последующие ноутбуки (04-10) будут грузить готовые `X_train`/`X_test` из `cache/`.

In [4]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
    shuffle=True,
)

print(f"X_train: {X_train_raw.shape}  | anomaly: {100 * y_train.mean():.2f}%")
print(f"X_test:  {X_test_raw.shape}  | anomaly: {100 * y_test.mean():.2f}%")
assert abs(y_train.mean() - y_test.mean()) < 1e-3, "Стратификация сломана"

split_indices = {
    "train_index": X_train_raw.index.tolist(),
    "test_index":  X_test_raw.index.tolist(),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
}
split_meta = {
    "n_train": len(X_train_raw),
    "n_test":  len(X_test_raw),
    "anomaly_rate_train": float(y_train.mean()),
    "anomaly_rate_test":  float(y_test.mean()),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
}
print(json.dumps(split_meta, indent=2))

X_train: (1477791, 42)  | anomaly: 23.63%
X_test:  (633340, 42)  | anomaly: 23.63%
{
  "n_train": 1477791,
  "n_test": 633340,
  "anomaly_rate_train": 0.23630202105710482,
  "anomaly_rate_test": 0.23630119682950707,
  "random_state": 42,
  "test_size": 0.3
}


## 2. Выбор колонок для log1p

Считается skewness на train-сете для 41 признака.

Применяется `log1p` к признакам, у которых `|skew|>2`

In [5]:
skew_train = X_train_raw[NUMERIC_FEATURES].skew()
skew_train_sorted = skew_train.abs().sort_values(ascending=False)

print("=== Skewness 41 числового признака на train ===")
for feat, val in skew_train_sorted.items():
    flag = " ← log1p" if val > 2 else ""
    print(f"  {feat:35s}  {skew_train[feat]:>10.2f}{flag}")

LOG1P_FEATURES = [f for f in NUMERIC_FEATURES if abs(skew_train[f]) > 2.0]
PASSTHROUGH_NUMERIC = [f for f in NUMERIC_FEATURES if f not in LOG1P_FEATURES]
print(f"\nlog1p применяется к {len(LOG1P_FEATURES)} колонкам:")
for f in LOG1P_FEATURES:
    print(f"  - {f}")
print(f"\nБез log1p ({len(PASSTHROUGH_NUMERIC)} колонок): "
      f"{PASSTHROUGH_NUMERIC}")

# Сохраним результат в табличку для отчёта
skew_table = pd.DataFrame({
    "feature": NUMERIC_FEATURES,
    "skew_train": [skew_train[f] for f in NUMERIC_FEATURES],
    "apply_log1p": [f in LOG1P_FEATURES for f in NUMERIC_FEATURES],
})
skew_table.to_csv(TAB_DIR / "preprocessing_skewness.csv", index=False)
print(f"\nСохранено: {TAB_DIR / 'preprocessing_skewness.csv'}")

=== Skewness 41 числового признака на train ===
  dst2src_urg_packets                     1215.64 ← log1p
  dst2src_bytes                            295.20 ← log1p
  src2dst_bytes                            273.31 ← log1p
  dst2src_psh_packets                      256.93 ← log1p
  src2dst_psh_packets                      250.17 ← log1p
  src2dst_urg_packets                      214.25 ← log1p
  bidirectional_urg_packets                213.95 ← log1p
  src2dst_packets                          209.59 ← log1p
  dst2src_packets                          203.63 ← log1p
  bidirectional_ack_packets                201.04 ← log1p
  bidirectional_psh_packets                167.34 ← log1p
  bidirectional_cwr_packets                 94.47 ← log1p
  bidirectional_min_piat_ms                 82.05 ← log1p
  bidirectional_ece_packets                 58.22 ← log1p
  src2dst_min_ps                            32.41 ← log1p
  bidirectional_rst_packets                 24.97 ← log1p
  bidirectional_min_ps  

## 3. Создание препроцессора (ColumnTransformer)

Создается из трех блоков:
1) `log1p` для `LOG1P_FEATURES` -> `FunctionTransformer(np.log1p, clip>=0)` -> `RobustScaler`
2) `scale_only` для `PASSTHROUGH_NUMBERIC` -> `RobuystScaler`
3) `passthrough` -> без изменений

`RobustScaler` использует медиану и IQR вместо среднего и std — устойчив
к выбросам, которые есть в каждой PS/PIAT/COUNTERS колонке. `with_centering=True`
(дефолт), `with_scaling=True`.

In [6]:
def safe_log1p(x):
    # np.log1p с защитой от отрицательных значений (clip к 0)
    return np.log1p(np.clip(x, 0, None))

log1p_then_scale = Pipeline([
    ("log1p", FunctionTransformer(safe_log1p, feature_names_out="one-to-one",
                                  validate=False)),
    ("scaler", RobustScaler()),
])

scale_only = Pipeline([
    ("scaler", RobustScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("log_scale",   log1p_then_scale, LOG1P_FEATURES),
        ("scale_only",  scale_only,       PASSTHROUGH_NUMERIC),
        ("passthrough", "passthrough",    [PROTOCOL_COL]),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform="pandas")
print(preprocessor)

ColumnTransformer(transformers=[('log_scale',
                                 Pipeline(steps=[('log1p',
                                                  FunctionTransformer(feature_names_out='one-to-one',
                                                                      func=<function safe_log1p at 0x731ba74e2d40>)),
                                                 ('scaler', RobustScaler())]),
                                 ['src2dst_packets', 'dst2src_packets',
                                  'src2dst_bytes', 'dst2src_bytes',
                                  'bidirectional_duration_ms',
                                  'bidirectional_min_ps',
                                  'bidirectional_max_ps',
                                  'bidirectional...
                                  'dst2src_mean_piat_ms',
                                  'dst2src_stddev_piat_ms',
                                  'dst2src_max_piat_ms', 'dst2src_min_piat_ms',
                           

In [7]:
print("Fit препроцессора на X_train_raw...")
X_train_pp = preprocessor.fit_transform(X_train_raw)
print(f"  X_train_pp: {X_train_pp.shape}, columns={list(X_train_pp.columns[:5])}...")

print("Transform на X_test_raw...")
X_test_pp = preprocessor.transform(X_test_raw)
print(f"  X_test_pp: {X_test_pp.shape}")

# Восстанавление порядока колонок в порядке MODEL_FEATURES
assert set(X_train_pp.columns) == set(MODEL_FEATURES), \
    f"Расхождение: лишние {set(X_train_pp.columns) - set(MODEL_FEATURES)}, " \
    f"пропали {set(MODEL_FEATURES) - set(X_train_pp.columns)}"
X_train_pp = X_train_pp[MODEL_FEATURES]
X_test_pp  = X_test_pp[MODEL_FEATURES]
assert list(X_train_pp.columns) == MODEL_FEATURES
assert list(X_test_pp.columns)  == MODEL_FEATURES
print("\nПорядок колонок совпадает с MODEL_FEATURES")

Fit препроцессора на X_train_raw...
  X_train_pp: (1477791, 42), columns=['src2dst_packets', 'dst2src_packets', 'src2dst_bytes', 'dst2src_bytes', 'bidirectional_duration_ms']...
Transform на X_test_raw...
  X_test_pp: (633340, 42)

Порядок колонок совпадает с MODEL_FEATURES


In [8]:
# 1. На train: медиана числовых scaled колонок должна быть = 0, IQR = 1
sample_cols = NUMERIC_FEATURES[:5]
print("=== Проверка скейлера (на train, первые 5 numeric колонок) ===")
for c in sample_cols:
    s = X_train_pp[c]
    print(f"  {c:35s} median={s.median():+.3f}  IQR={s.quantile(.75) - s.quantile(.25):.3f}")

# 2. protocol не изменился — значения те же, что и в исходном X_train_raw
proto_unchanged = (X_train_pp[PROTOCOL_COL].values
                   == X_train_raw[PROTOCOL_COL].values).all()
print(f"\nprotocol сохранён без изменений: {proto_unchanged}")
print(f"  уникальные значения protocol: "
      f"{sorted(X_train_pp[PROTOCOL_COL].unique().tolist())}")

# 3. Никаких NaN/Inf
n_nan_train = X_train_pp.isna().sum().sum()
n_inf_train = np.isinf(X_train_pp.values).sum()
n_nan_test  = X_test_pp.isna().sum().sum()
n_inf_test  = np.isinf(X_test_pp.values).sum()
print(f"\nNaN/Inf после препроцессинга:")
print(f"  train: NaN={n_nan_train}, Inf={n_inf_train}")
print(f"  test:  NaN={n_nan_test}, Inf={n_inf_test}")
assert n_nan_train == 0 and n_inf_train == 0
assert n_nan_test  == 0 and n_inf_test  == 0
print("\nВсе проверки пройдены")

=== Проверка скейлера (на train, первые 5 numeric колонок) ===
  src2dst_packets                     median=+0.000  IQR=1.000
  dst2src_packets                     median=+0.000  IQR=1.000
  src2dst_bytes                       median=+0.000  IQR=1.000
  dst2src_bytes                       median=+0.000  IQR=1.000
  bidirectional_duration_ms           median=+0.000  IQR=1.000

protocol сохранён без изменений: True
  уникальные значения protocol: [6.0, 17.0]

NaN/Inf после препроцессинга:
  train: NaN=0, Inf=0
  test:  NaN=0, Inf=0

Все проверки пройдены


In [9]:
# 1. X_train, X_test, y_train, y_test -> parquet
X_train_pp.to_parquet(CACHE_DIR / "X_train.parquet", index=False)
X_test_pp.to_parquet (CACHE_DIR / "X_test.parquet",  index=False)
# y_train/y_test как DataFrame с одним столбцом 'target'
pd.DataFrame({"target": y_train.values}).to_parquet(
    CACHE_DIR / "y_train.parquet", index=False)
pd.DataFrame({"target": y_test.values}).to_parquet(
    CACHE_DIR / "y_test.parquet",  index=False)

# 2. preprocessor -> joblib (sklearn-объект)
joblib.dump(preprocessor, ARTIFACTS / "preprocessor.joblib", compress=3)

# 3. preprocessing_config -> JSON для документации и валидации
preprocessing_config = {
    "sklearn_version": sklearn.__version__,
    "model_features":  MODEL_FEATURES,
    "log1p_features":  LOG1P_FEATURES,
    "scale_only_features": PASSTHROUGH_NUMERIC,
    "passthrough_features": [PROTOCOL_COL],
    "skewness_threshold_for_log1p": 2.0,
    "scaler":          "RobustScaler",
    "scaler_params":   {"with_centering": True, "with_scaling": True,
                        "quantile_range": (25.0, 75.0)},
    "split": {
        "method":       "stratified_random",
        "test_size":    TEST_SIZE,
        "random_state": RANDOM_STATE,
        "n_train":      int(len(y_train)),
        "n_test":       int(len(y_test)),
        "anomaly_rate_train": float(y_train.mean()),
        "anomaly_rate_test":  float(y_test.mean()),
    },
}
with (ARTIFACTS / "preprocessing_config.json").open("w") as f:
    json.dump(preprocessing_config, f, indent=2, ensure_ascii=False)

# 4. feature_names.json — финальные 42 имени в порядке столбцов
with (ARTIFACTS / "feature_names.json").open("w") as f:
    json.dump({"feature_names": MODEL_FEATURES,
               "n_features": 42}, f, indent=2, ensure_ascii=False)

# 5. Скопируем split_meta тоже в JSON
with (ARTIFACTS / "split_meta.json").open("w") as f:
    json.dump(split_meta, f, indent=2)

print("Сохранено:")
for p in [
    CACHE_DIR / "X_train.parquet", CACHE_DIR / "X_test.parquet",
    CACHE_DIR / "y_train.parquet", CACHE_DIR / "y_test.parquet",
    ARTIFACTS / "preprocessor.joblib",
    ARTIFACTS / "preprocessing_config.json",
    ARTIFACTS / "feature_names.json",
    ARTIFACTS / "split_meta.json",
]:
    size_mb = p.stat().st_size / 1024**2
    print(f"  {p.relative_to(PROJECT_ROOT)}  ({size_mb:.2f} MB)")

Сохранено:
  cache/X_train.parquet  (93.89 MB)
  cache/X_test.parquet  (39.58 MB)
  cache/y_train.parquet  (0.21 MB)
  cache/y_test.parquet  (0.09 MB)
  artifacts/preprocessor.joblib  (0.00 MB)
  artifacts/preprocessing_config.json  (0.00 MB)
  artifacts/feature_names.json  (0.00 MB)
  artifacts/split_meta.json  (0.00 MB)


## 4. Round-trip проверка артифактов

Загрузка препроцессора обратно из joblib, прогон сырного test через него. Сравнение с уже загруденной `X_test.parquet`.

- Если расхождений нет, артефак был простроен верно

In [10]:
# Сброс переменных
del preprocessor, X_train_pp, X_test_pp

# 1. Препроцессор
pp_reload = joblib.load(ARTIFACTS / "preprocessor.joblib")

# 2. Прогон сырого X_test через перезагруженный препроцессор
X_test_via_pp = pp_reload.transform(X_test_raw)[MODEL_FEATURES]

# 3. Загруженный X_test
X_test_loaded = pd.read_parquet(CACHE_DIR / "X_test.parquet")

# 4. Сравнение
diff = (X_test_via_pp.values - X_test_loaded.values)
max_abs_diff = float(np.abs(diff).max())
print(f"max |X_test_via_pp - X_test_loaded|: {max_abs_diff:.2e}")
assert max_abs_diff < 1e-9, "Round-trip провалился"

# 5. y_train/y_test тоже round-trip
y_train_loaded = pd.read_parquet(CACHE_DIR / "y_train.parquet")["target"].values
y_test_loaded  = pd.read_parquet(CACHE_DIR / "y_test.parquet" )["target"].values
assert np.array_equal(y_train_loaded, y_train.values)
assert np.array_equal(y_test_loaded,  y_test.values)

print("Round-trip пройден. Артефакты пригодны для всех следующих ноутбуков.")

max |X_test_via_pp - X_test_loaded|: 0.00e+00
Round-trip пройден. Артефакты пригодны для всех следующих ноутбуков.
